In [ ]:
import os
import dotenv
from logfire.query_client import LogfireQueryClient

# Load credentials
dotenv.load_dotenv("/workspaces/.codespaces/shared/.env")

# Create Logfire query client
read_token = os.getenv("LOGFIRE_READ_TOKEN")
logfire_query_client = LogfireQueryClient(
    read_token=read_token
)

print("Logfire Query Client initialized")

In [ ]:
from datetime import datetime, timedelta, timezone

trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'agent run'
       OR span_name LIKE 'invoke_agent %'
    ORDER BY start_timestamp DESC
    LIMIT 5
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

trace_rows["rows"]

In [ ]:
trace_id = trace_rows["rows"][0]["trace_id"]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' AS all_messages,
        attributes->>'gen_ai.usage.input_tokens' AS input_tokens,
        attributes->>'gen_ai.usage.output_tokens' AS output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND (
          span_name = 'agent run'
          OR span_name LIKE 'invoke_agent %'
      )
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

print("Agent runs:", len(run_row["rows"]))
print("Input tokens:", [
    row["input_tokens"] for row in run_row["rows"]
])
print("Output tokens:", [
    row["output_tokens"] for row in run_row["rows"]
])
print("Messages per run:", [
    len(row["all_messages"] or []) for row in run_row["rows"]
])

In [ ]:
attributes_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT attributes
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    LIMIT 1
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

if attributes_result["rows"]:
    print(list(attributes_result["rows"][0]["attributes"].keys()))
else:
    print("No matching records")

In [ ]:
usage_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->>'gen_ai.aggregated_usage.input_tokens'
            AS input_tokens,
        attributes->>'gen_ai.aggregated_usage.output_tokens'
            AS output_tokens,
        attributes->>'gen_ai.aggregated_usage.cache_read.input_tokens'
            AS cached_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

usage_result["rows"]

In [ ]:
evaluation_rows = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->'final_result' AS final_result
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    print(f"Run {i}:")
    print("Result type:", type(result).__name__)
    print("Fields:", list(result.keys()) if isinstance(result, dict) else "Not a dictionary")
    print()

In [ ]:
evaluation_data = []

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]

    evaluation_data.append({
        "run": i,
        "found_answer": result.get("found_answer"),
        "confidence": result.get("confidence"),
        "answer_type": result.get("answer_type"),
        "references_count": len(result.get("references") or []),
        "answer_length": len(result.get("answer") or ""),
    })

evaluation_data

In [ ]:
for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    print(f"Run {i}: {result.get('answer', '')[:150]!r}")

In [ ]:
scrub_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->'logfire.scrubbed' AS scrubbed
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

scrub_result["rows"]

In [ ]:
evaluation_dataset = []

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    answer = result.get("answer", "")

    is_scrubbed = (
        answer.startswith("[Scrubbed due to")
    )

    evaluation_dataset.append({
        "run": i,
        "answer": None if is_scrubbed else answer,
        "found_answer": result.get("found_answer"),
        "confidence": result.get("confidence"),
        "references_count": len(result.get("references") or []),
        "scrubbed": is_scrubbed,
        "eligible_for_text_evaluation": not is_scrubbed,
    })

for item in evaluation_dataset:
    print({
        key: value
        for key, value in item.items()
        if key != "answer"
    })